# Workflow Patterns in Pure Python

**What you will build:** the four classic workflow patterns — chaining, routing, parallelization, and orchestrator-workers — each in a few lines of plain Python. The lesson underneath: for most tasks you **don't need an agent, or a framework**. You saw the model drive in 0.3; here *you* drive.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/04_workflow_patterns.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    """One-shot helper: system + user in, text out."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready. Model:", MODEL)

## Agent vs workflow, in one line of code each

In 0.3 the model chose the steps. In a workflow, *you* write the steps as ordinary control flow. These patterns come from Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents); this is the same material as chapters 04a–04c of the n8n course, but here a "node" is just a function call.

```{note}
Rule of thumb from 0.0: reach for the simplest thing that works. If you can draw the flowchart in advance, it's a workflow — write the `if`s yourself. Save agents for when the path can't be known ahead of time.
```

## Pattern 1 — Prompt chaining

Break one hard task into a sequence where each step's output feeds the next. Each step is simpler, so each is more reliable. Here: a raw recipe becomes a beginner version, then a shopping list.

In [ ]:
recipe = "Beef lasagna: brown 500g beef, layer with tomato sauce, bechamel and pasta, bake 40 min."

simplified = ask("You simplify recipes for absolute beginners. Number every step.", recipe)
shopping   = ask("You are a shopping assistant. Output ONLY a grouped shopping list.", simplified)

print("SIMPLIFIED:\n", simplified[:300], "...\n")
print("SHOPPING LIST:\n", shopping)

That's the whole pattern: `output_1 -> input_2`. No framework, no agent — just two calls in a row.

## Pattern 2 — Routing

Classify the input first, then send it down a different path. This is where structured output (0.2) pays off: the classification decides the branch.

In [ ]:
def route_ticket(ticket):
    category = ask(
        "Classify the support ticket as exactly one word: billing, technical, or account.",
        ticket).strip().lower()

    # YOU control the branch — plain Python:
    if "billing" in category:
        return ask("You are a billing specialist. Be precise about charges and refunds.", ticket)
    elif "technical" in category:
        return ask("You are a technical support engineer. Give clear troubleshooting steps.", ticket)
    else:
        return ask("You are an account specialist. Help with login and settings.", ticket)

print(route_ticket("I was double-charged for my plan this month."))

## Pattern 3 — Parallelization

When subtasks are independent, run them at the same time instead of one after another. With `AsyncOpenAI` and `asyncio.gather`, three calls take about as long as one. (Notebooks support top-level `await` directly.)

In [ ]:
import asyncio
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

async def analyze(aspect, text):
    r = await aclient.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role":"system","content":f"In one sentence, analyze the {aspect} of the review."},
                  {"role":"user","content":text}])
    return aspect, r.choices[0].message.content

review = "The laptop is fast and beautiful, but it runs hot and the battery barely lasts three hours."

# Fire all three at once and wait for all of them:
results = await asyncio.gather(
    analyze("sentiment", review),
    analyze("hardware quality", review),
    analyze("battery", review),
)
for aspect, out in results:
    print(f"{aspect:16}: {out}")

## Pattern 4 — Orchestrator-workers

A coordinator LLM breaks the task into subtasks, workers handle each (often in parallel), and a final step combines them. This is the workflow that looks most like an agent — but *you* still fix the shape (plan, then work, then synthesize).

In [ ]:
from pydantic import BaseModel

class Plan(BaseModel):
    subtopics: list[str]

def orchestrate(topic):
    # 1. Orchestrator: plan the subtopics (structured output from 0.2)
    raw = client.chat.completions.create(
        model=MODEL, temperature=0, response_format={"type":"json_object"},
        messages=[{"role":"system","content":"Break the topic into 3 subtopics. JSON: {\"subtopics\": [...]}"},
                  {"role":"user","content":topic}])
    plan = Plan.model_validate_json(raw.choices[0].message.content)

    # 2. Workers: one short paragraph per subtopic
    parts = [ask("Write one concise paragraph.", f"Topic: {topic}\nSubtopic: {s}") for s in plan.subtopics]

    # 3. Synthesize
    return ask("Combine these paragraphs into a coherent short article.", "\n\n".join(parts))

print(orchestrate("Why bees matter to agriculture"))

::::{dropdown} 🔍 The four patterns side by side
:color: info

```
Chaining         A -> B -> C            each step feeds the next
Routing          classify -> A | B | C  one branch runs, chosen by a classifier
Parallelization  A, B, C at once        independent subtasks, combined at the end
Orchestrator     plan -> [workers] ->   an LLM plans the subtasks, workers execute,
                 synthesize             a final step merges them
```

All four have a shape *you* decided in advance. That's what makes them workflows. The moment you can't draw the shape — because it depends on what the model discovers mid-task — you need the agent loop from 0.3.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Extend the chain.** Add a third step to Pattern 1 that translates the shopping list into Spanish.
2. **Add a route.** Give `route_ticket` a fourth category, `sales`, with its own specialist prompt.
3. **Time it.** Wrap Pattern 3 in `import time; t=time.time(); ... ; print(time.time()-t)`, then rewrite it as three sequential `await` calls and compare. Feel why parallelization exists.
::::

**What's next:** in **0.5** we add a loop *back* into a workflow — the **reflection** pattern (generate, critique, improve) — and pause a workflow for **human approval**.